# Assignment 5

In this assignment, you will work with a slightly processed version of the Student Performance dataset from the UCI Machine Learning repository. Please check out this link to familiarize yourself with the dataset: https://archive.ics.uci.edu/dataset/320/student+performance

Please add code and text chunks as necessary throughout the document in the relevant sections.

The task is to use machine learning to predict student performance (recorded in a the `G3` variable as a score between 0 and 20) using a set of predictors / features.

## Preprocessing on Data:

The data you will get is a version in which categorical variables have been transformed into individual dummy variables (1 and 0) in order to make working with them easier. For example, the variable `guardian` which has values `father`, `mother`, or `other` in the original dataset has been transformed into two dummy variables, `guardian_mother` and `guardian_other`, both assuming values of 1 and 0, with `father` as the excluded category (because the other two columns already capture the information in what would have been `guardian_father`). Similarly, the `internet` variable, which has values of `yes` and `no` in the original dataset has been transformed into `internet_yes`, which assumes the value of `1` for `yes` and `0` for `no`. All others variables that are originally categorical have been transformed to numeric in the same way. There are variables in the dataset that are ordinal level, such as `Medu` for mother's education and `Fedu` for father's education, but they are already recorded numerically. You can treat them as numeric variables (no need to transform them.)

## Target Variables:

The dataset has three potential target variables (i.e. outcomes to predict): `G1`, `G2` and `G3.` **Please do not use `G1` or `G2`; neither as predictors, nor as outcome.** Use `G3` as the outcome to predict, and every variable other than `G1`, `G2`, and `G3` as predictors.

## Load Dataset:

The dataset is stored in a CSV file. Please run the following code chunk to load it as a Pandas DataFrame.


In [1]:
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/omerfyalcin/colab-data/refs/heads/main/student_performance.csv')

## Task 1 (1 point)

Using the dataframe, `df`, and based on the information given above, create a dataframe for predictors (`X`), and a target column (outcome variable, `y`). Then, divide `X` and `y` into training and test sets.

In [2]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['G1', 'G2', 'G3'])
y = df['G3']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((519, 39), (130, 39), (519,), (130,))

## Task 2 (3 points)

Fit three models with the training data:

- Linear Regression (https://scikit-learn.org/1.5/modules/generated/sklearn.linear_model.LinearRegression.html)
- Decision Tree Regression (https://scikit-learn.org/1.5/modules/generated/sklearn.tree.DecisionTreeRegressor.html)
- Random Forest (https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)


For linear regression and decision tree, use the default values for all arguments. For random forest, set `max_features='sqrt'`.

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Initialize models
lr_model = LinearRegression()
dt_model = DecisionTreeRegressor()
rf_model = RandomForestRegressor(max_features='sqrt', random_state=42)

# Fit models on training data
lr_model.fit(X_train, y_train)
dt_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

RandomForestRegressor(max_features='sqrt', random_state=42)

## Task 3 (3 points)

Report the training set and test performance of each model, using root mean squared error as the evaluation metric.

Considering our class dicussion on overfitting vs. underfitting, which of the three models provides the best fit? Put another way, which model would do the best job in predicting student performance in data it has not "seen" before.

Does any model overfit? If so, which model? Explain.

(Our class discussion should be enough, but if you need a review of the topic, read Section 2.2.1 "Measuring the Quality of Fit" from Introduction to Statistical Learning, downloadable at https://www.statlearning.com/)


In [4]:
from sklearn.metrics import mean_squared_error
import numpy as np

def rmse(model, X_train, X_test, y_train, y_test):
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)

    train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))

    return train_rmse, test_rmse

lr_train_rmse, lr_test_rmse = rmse(lr_model, X_train, X_test, y_train, y_test)
dt_train_rmse, dt_test_rmse = rmse(dt_model, X_train, X_test, y_train, y_test)
rf_train_rmse, rf_test_rmse = rmse(rf_model, X_train, X_test, y_train, y_test)

print(f"Linear Regression RMSE - Train: {lr_train_rmse:.2f}, Test: {lr_test_rmse:.2f}")
print(f"Decision Tree RMSE - Train: {dt_train_rmse:.2f}, Test: {dt_test_rmse:.2f}")
print(f"Random Forest RMSE - Train: {rf_train_rmse:.2f}, Test: {rf_test_rmse:.2f}")

Linear Regression RMSE - Train: 2.54, Test: 2.86
Decision Tree RMSE - Train: 0.00, Test: 3.74
Random Forest RMSE - Train: 1.00, Test: 2.78




---


Overfitting Analysis:

1. Linear Regression

*   Train RMSE: 2.54
*   Test RMSE: 2.86

**Observation**: The training and test errors are fairly close. This suggests that Linear Regression does not overfit and generalizes reasonably well. However, the relatively high RMSE indicates that the model may underfit, meaning it might not capture the complexity of the data well.

---

2. Decision Tree
*   Train RMSE: 0.00
*   Test RMSE: 3.74

**Observation**: The training RMSE is 0.00, meaning the Decision Tree perfectly memorized the training data. However, the test RMSE is 3.74, which is significantly higher. This is a classic case of overfitting—the model is too complex and fits the training data too well, but fails to generalize to unseen data. Decision Tree clearly overfits.

---

3. Random Forest
*   Train RMSE: 1.00
*   Test RMSE: 2.78

**Observation**: The training error is lower than Linear Regression but not as extreme as the Decision Tree. The test error is also lower than both models, meaning it generalizes better. Some overfitting is present, but it is less severe than Decision Tree. Random Forest is the best-performing model and generalizes the best.


---


Best Model: Random Forest (good tradeoff between fit and generalization).


---



## Task 4 (3 points)



The Random Forest algorithm combines the predictions from many decision trees when making its prediction. A regular decision tree would consider all available variables (features) when deciding on the best way to split a branch. In random forests, in every split or every decision tree, only a random subset of the variables are considered. The `max_features` parameter you used in Task 2 determines what the size of this random subset is going to be. For example, `sqrt` means that the number will be the square root of the number of total predictors. For example, if you had 20 total predictors, then 4 predictors would be considered at each split because sqrt(20) is approximately 4.472, which is rounded to 4. (In this dataset you have more than 20 predictors, so the number must have been calculated accordingly.) RandomForestRegressor's `max_features` argument can also manually be set to an integer.

In this task, use cross validation to decide the optimal value of `max_features`. You can do the cross validation on the training set only. In particular, consider all integers in the 1 to 15 range for `max_features`.

- Based on root mean squared error as the evaluation metric, what value gives the best result in cross validation?

- Do you think `sqrt` was a good value for `max_features` or did you find another value to be better?

In [5]:
from sklearn.model_selection import cross_val_score

max_features_range = range(1, 16)
cv_rmse_scores = []

for max_f in max_features_range:
    rf = RandomForestRegressor(max_features=max_f, random_state=42)
    rmse_score = -cross_val_score(rf, X_train, y_train, scoring='neg_root_mean_squared_error', cv=5).mean()
    cv_rmse_scores.append(rmse_score)

best_max_features = max_features_range[np.argmin(cv_rmse_scores)]
print(f"Best max_features: {best_max_features}")

Best max_features: 9


Analysis of max_features Selection:

1. **Best Value Based on RMSE** (Cross-Validation)-> The best value for max_features found through cross-validation is 9. This means that when the Random Forest model was trained with max_features=9, it produced the lowest root mean squared error (RMSE), indicating the best predictive performance.

2. **Was sqrt a Good Choice?** -> Initially, we set max_features='sqrt'. Since the dataset contains more than 81 features, sqrt(81) ≈ 9, which is actually very close to the best value found through cross-validation. This suggests that sqrt was already a reasonable default choice. However, tuning the hyperparameter manually confirmed that 9 is the optimal value, slightly refining the model’s performance. sqrt was a good choice, but explicitly setting max_features=9 is slightly better for this dataset.

By manually tuning, we were able to confirm that sqrt was a solid default, but a small adjustment improved the model’s predictive accuracy.